<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_Kids_Middle_School_Slopes_Rise_Over_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understanding Slope through Stair Steepness

This notebook contains the code to generate an educational animation designed for middle school students. The visualization uses the physical metaphor of climbing stairs to introduce the mathematical concept of slope.

## Learning Objectives

By the end of this lesson/video, students should be able to:
1. Define slope as a measure of steepness.
2. Identify the horizontal change as 'Run'.
3. Identify the vertical change as 'Rise'.
4. Understand that slope is calculated as the ratio of Rise over Run (m = rise/run).
5. Compare different slopes and understand that a larger numerical value represents a steeper incline.

## Key Concepts Covered

### 1. The Physical Metaphor
Stairs provide a concrete example of slope. A shallow staircase is easy to climb (gentle slope), while a tall, narrow staircase requires more effort (steep slope).

### 2. Rise over Run
* **Rise**: The vertical distance between steps (change in y).
* **Run**: The horizontal distance of the step tread (change in x).
* **The Formula**: Slope (m) = Rise / Run.

### 3. Comparing Magnitudes
* **Gentle Slope**: Example shown with Rise=1, Run=2. Calculation: 1/2 = 0.5.
* **Steep Slope**: Example shown with Rise=3, Run=1. Calculation: 3/1 = 3.
* **Comparison**: Since 3 > 0.5, the second staircase is mathematically steeper.

## Guide for Educators and Parents

* **Pause and Reflect**: Encourage students to pause the video during the 'Which is steeper?' prompt to make their own predictions before the math is revealed.
* **Real-world Application**: Ask students to look at ramps, hills, or roofs in their neighborhood and estimate if they have a 'large' or 'small' slope based on the rise and run.
* **Variable Modification**: Advanced students can modify the `GENTLE` and `STEEP` dictionaries in the code below to see how changing the numbers affects the generated animation.

In [2]:
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science dvisvgm
!pip install "manim>=0.18.0" "numpy<2.0.0"

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,299 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,129 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,974 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.3 M

In [1]:
from manim import *
%load_ext manim

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


The manim module is not an IPython extension.


In [4]:
%%manim -v WARNING SlopeStairs

"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
Level: Middle School
Concept: Slope as stair steepness (rise over run)
Output: 1080x1920 (9:16 portrait), 30 fps, ~27.6 s, ~2.7 MB
"""
import numpy as np
from manim import *
import manimpango

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
# Portrait 9:16 frame. Manim keeps frame_width fixed and derives frame_height
# from the pixel aspect, so a 1080x1920 render would otherwise default to a
# ~14.2 x 25.3 world (everything tiny). Pin frame_width so the world is exactly
# 4.5 x 8 (== 9:16) and the layout fills the canvas.
config.frame_width = 4.5
config.frame_height = 8.0
# Portrait pixel canvas (9:16). Match the world aspect so cells stay square.
config.pixel_width = 1080
config.pixel_height = 1920
config.frame_rate = 30

# Clean sans-serif (Inter/Roboto per brief) with a robust fallback.
_AVAIL = set(manimpango.list_fonts())
FONT = next((f for f in ("Inter", "Roboto", "DejaVu Sans") if f in _AVAIL), None)

PALETTE = {
    "paper":   "#FFFFFF",
    "grid":    "#E0E0E0",
    "sage":    "#86A788",
    "coral":   "#E5989B",
    "mustard": "#E9C46A",
    "navy":    "#2A3D66",
    "red":     "#D62828",
}
INK    = PALETTE["navy"]
RISE_C = PALETTE["coral"]
RUN_C  = PALETTE["sage"]

GRID_X, GRID_Y = 4, 4
PLANE_X, PLANE_Y = 3.7, 3.7          # square cells (3.7/4 each axis)
PLANE_SHIFT = UP * 0.45

# ---- Parametric: tweak these two dicts to experiment with any slope ----
GENTLE = dict(run=2, rise=1, value="0.5", note="Gentle climb  -  small slope")
STEEP  = dict(run=1, rise=3, value="3",   note="Steep climb  -  big slope!")

CLIMBER_H = 0.55
AUTHOR_WM = "(c) Mugambi Ndwiga  /  @craftsandengineering"

TIME = dict(intro=1.0, hook=1.0, settle=0.7, plane=1.2,
            build=0.9, line=0.7, braces=1.0,
            climb_gentle=2.0, climb_steep=2.6, card=0.7, hold=1.0, swap=0.6,
            compare=1.0, compare_card=0.7, compare_hold=1.1, close=1.0)


# ----------------------------------------------------------------------
# HELPERS
# ----------------------------------------------------------------------
def label(txt, size=24, color=INK, bold=False):
    kw = dict(color=color, font_size=size, font=FONT)
    if bold:
        kw["weight"] = BOLD
    return Text(txt, **kw)

def dotted(txt, dot_color, size=24, s=1.0):
    t = label(txt, size, INK, bold=True).scale(s)
    d = Dot(radius=0.06, color=dot_color)
    return VGroup(d, t).arrange(RIGHT, buff=0.10)

def make_badge():
    inner = VGroup(label("FOR KIDS", 16, WHITE, bold=True),
                   label("Middle School", 20, WHITE, bold=True)
                   ).arrange(DOWN, buff=0.05)
    box = RoundedRectangle(corner_radius=0.12,
                           width=inner.width + 0.32, height=inner.height + 0.24,
                           fill_color=PALETTE["navy"], fill_opacity=0.80,
                           stroke_width=0)
    return VGroup(box, inner).to_corner(UR, buff=0.20)

def make_watermark():
    wm = label(AUTHOR_WM, 15, INK)
    wm.set_opacity(0.60).to_corner(DL, buff=0.16)
    return wm

def make_climber(color=INK, height=CLIMBER_H):
    body = Line([0, 0.10, 0], [0, -0.18, 0], color=color, stroke_width=6)
    head = Dot(point=[0, 0.225, 0], radius=0.09, color=color)
    arms = Line([-0.14, 0.02, 0], [0.14, 0.02, 0], color=color, stroke_width=6)
    legL = Line([0, -0.18, 0], [-0.12, -0.44, 0], color=color, stroke_width=6)
    legR = Line([0, -0.18, 0], [0.12, -0.44, 0], color=color, stroke_width=6)
    g = VGroup(head, body, arms, legL, legR)
    g.scale_to_fit_height(height)
    return g

def make_card(width, height):
    return RoundedRectangle(corner_radius=0.16, width=width, height=height,
                            stroke_color=INK, stroke_width=2.5,
                            fill_color=PALETTE["paper"], fill_opacity=1.0)

def fraction(num_tok, den_tok):
    bar = Line(LEFT, RIGHT, color=INK, stroke_width=3
               ).set_length(max(num_tok.width, den_tok.width) * 1.18)
    return VGroup(num_tok, bar, den_tok).arrange(DOWN, buff=0.07)


# ----------------------------------------------------------------------
# SCENE
# ----------------------------------------------------------------------
class SlopeStairs(Scene):
    def construct(self):
        self.camera.background_color = PALETTE["paper"]
        self.watermark = make_watermark()
        self.add(self.watermark)
        self.badge = make_badge()

        self.intro()
        self.plane_group = self.build_plane()
        self.show_case(GENTLE, climb_time=TIME["climb_gentle"])
        self.show_case(STEEP,  climb_time=TIME["climb_steep"])
        self.compare()
        self.outro()

    def intro(self):
        big = VGroup(label("SLOPE", 60, INK, bold=True),
                     label("as Stair Steepness", 30, INK)
                     ).arrange(DOWN, buff=0.16).move_to(UP * 0.9)
        hook = VGroup(label("Which line is", 28, INK, bold=True),
                      label("harder to climb?", 28, INK, bold=True)
                      ).arrange(DOWN, buff=0.14).move_to(DOWN * 0.4)
        self.play(FadeIn(self.badge, shift=DOWN * 0.2),
                  FadeIn(big, shift=UP * 0.2), run_time=TIME["intro"])
        self.play(Write(hook), run_time=TIME["hook"])
        self.wait(TIME["settle"])
        self.small_title = label("SLOPE", 26, INK, bold=True
                                 ).to_corner(UL, buff=0.22)
        self.play(FadeOut(big), FadeOut(hook),
                  FadeIn(self.small_title), run_time=0.7)

    def build_plane(self):
        plane = NumberPlane(
            x_range=[0, GRID_X, 1], y_range=[0, GRID_Y, 1],
            x_length=PLANE_X, y_length=PLANE_Y,
            background_line_style={"stroke_color": PALETTE["grid"],
                                   "stroke_width": 1.5, "stroke_opacity": 1.0},
            axis_config={"stroke_color": INK, "stroke_width": 3,
                         "include_ticks": False, "include_tip": True,
                         "tip_width": 0.16, "tip_height": 0.16},
        ).shift(PLANE_SHIFT)
        xl = label("x", 26, bold=True).next_to(plane.x_axis.get_end(), RIGHT, buff=0.08)
        yl = label("y", 26, bold=True).next_to(plane.y_axis.get_end(), UP, buff=0.08)
        zero = label("0", 20).set_opacity(0.65).next_to(plane.c2p(0, 0), DL, buff=0.06)
        self.plane = plane
        self.play(Create(plane), FadeIn(xl), FadeIn(yl), FadeIn(zero),
                  run_time=TIME["plane"])
        return VGroup(plane, xl, yl, zero)

    def show_case(self, cfg, climb_time):
        p = self.plane
        run, rise = cfg["run"], cfg["rise"]

        # The rise x run cell (one stair), shaded
        cell = Polygon(p.c2p(0, 0), p.c2p(run, 0), p.c2p(run, rise), p.c2p(0, rise),
                       fill_color=PALETTE["mustard"], fill_opacity=0.55,
                       stroke_color=INK, stroke_width=2)

        # THE LINE (the premise): diagonal of the cell = slope
        the_line = Line(p.c2p(0, 0), p.c2p(run, rise), color=INK, stroke_width=6)

        # rise/run braces (navy braces, colour carried by dots on labels)
        run_seg  = Line(p.c2p(0, 0),   p.c2p(run, 0))
        rise_seg = Line(p.c2p(run, 0), p.c2p(run, rise))
        run_br  = Brace(run_seg,  DOWN,  color=RUN_C,  buff=0.06)
        rise_br = Brace(rise_seg, RIGHT, color=RISE_C, buff=0.06)
        run_lbl  = dotted(f"run = {run}",  RUN_C,  size=24
                          ).next_to(run_br, DOWN, buff=0.10)
        rise_lbl = dotted(f"rise = {rise}", RISE_C, size=24
                          ).next_to(rise_br, RIGHT, buff=0.10)

        # Climber ascends the line (feet on the line, body on upper side)
        P0, P1 = p.c2p(0, 0), p.c2p(run, rise)
        d = P1 - P0
        nrm = np.array([-d[1], d[0], 0.0])
        nrm = nrm / np.linalg.norm(nrm)
        off = nrm * (CLIMBER_H * 0.5)
        climber = make_climber()
        walk = Line(P0 + off, P1 + off)
        climber.move_to(P0 + off)

        # Equation card
        card = make_card(4.35, 1.42).to_edge(DOWN, buff=0.46)
        eq = VGroup(
            label("slope", 26, INK, bold=True).scale(0.7),
            label("=", 26, INK, bold=True).scale(0.7),
            fraction(dotted("rise", RISE_C, size=26, s=0.62),
                     dotted("run",  RUN_C,  size=26, s=0.62)),
            label("=", 26, INK, bold=True).scale(0.7),
            fraction(dotted(str(rise), RISE_C, size=26, s=0.62),
                     dotted(str(run),  RUN_C,  size=26, s=0.62)),
            label("=", 26, INK, bold=True).scale(0.7),
            label(cfg["value"], 30, INK, bold=True),
        ).arrange(RIGHT, buff=0.16)
        eq.scale_to_fit_width(card.width - 0.5)
        note = label(cfg["note"], 22, INK, bold=True)
        content = VGroup(eq, note).arrange(DOWN, buff=0.2).move_to(card)

        case = VGroup(cell, the_line, run_br, rise_br,
                      run_lbl, rise_lbl, climber, card, content)

        self.play(DrawBorderThenFill(cell), run_time=TIME["build"])
        self.play(Create(the_line), run_time=TIME["line"])
        self.play(GrowFromCenter(run_br), GrowFromCenter(rise_br),
                  FadeIn(run_lbl), FadeIn(rise_lbl), run_time=TIME["braces"])
        self.play(FadeIn(climber), run_time=0.4)
        self.play(MoveAlongPath(climber, walk),
                  run_time=climb_time, rate_func=linear)
        self.play(FadeIn(card), run_time=TIME["card"])
        self.play(FadeIn(content, shift=UP * 0.1), run_time=0.6)
        self.wait(TIME["hold"])
        self.play(FadeOut(case), run_time=TIME["swap"])

    def compare(self):
        p = self.plane
        ray_shallow = Line(p.c2p(0, 0), p.c2p(4, 2), color=INK, stroke_width=6)
        ray_steep   = Line(p.c2p(0, 0), p.c2p(1, 3), color=INK, stroke_width=6)
        d_sh = Dot(p.c2p(4, 2), radius=0.07, color=RUN_C)
        d_st = Dot(p.c2p(1, 3), radius=0.07, color=RISE_C)

        card = make_card(4.35, 1.7).to_edge(DOWN, buff=0.45)
        lines = VGroup(
            dotted("shallow line:  slope = 0.5", RUN_C,  size=24),
            dotted("steep line:    slope = 3",   RISE_C, size=24),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.16)
        msg = label("Bigger slope = steeper!", 26, INK, bold=True)
        content = VGroup(lines, msg).arrange(DOWN, buff=0.2).move_to(card)
        bundle = VGroup(ray_shallow, ray_steep, d_sh, d_st, card, content)

        self.play(Create(ray_shallow), Create(ray_steep), run_time=TIME["compare"])
        self.play(FadeIn(d_sh), FadeIn(d_st), run_time=0.3)
        self.play(FadeIn(card), FadeIn(content), run_time=TIME["compare_card"])
        self.wait(TIME["compare_hold"])
        # Keep the scene on screen; the outro wipe transitions out of it
        # (no fade-to-white gap). Stash refs so outro can clear them.
        self.compare_bundle = bundle

    def outro(self):
        fw, fh = config.frame_width, config.frame_height

        # Solid navy card that slides UP from below to wipe the screen.
        # An opaque rect sliding in never looks washed-out (unlike a fade).
        cover = Rectangle(width=fw + 0.4, height=fh + 0.4,
                          fill_color=PALETTE["navy"], fill_opacity=1.0,
                          stroke_width=0)
        cover.move_to(DOWN * (fh / 2 + cover.height / 2 + 0.2))   # fully off-screen
        self.play(cover.animate.move_to(ORIGIN),
                  self.watermark.animate.set_opacity(0),
                  run_time=0.8, rate_func=smooth)
        # Everything underneath is now hidden; drop it from the scene.
        self.remove(self.compare_bundle, self.plane_group,
                    self.small_title, self.badge)

        # Slope motif accent (callback to the topic): a small rising line + dot.
        motif = Line(LEFT * 0.55 + DOWN * 0.32, RIGHT * 0.55 + UP * 0.32,
                     color=PALETTE["mustard"], stroke_width=7)
        tip = Dot(motif.get_end(), radius=0.075, color=WHITE)
        accent = VGroup(motif, tip)

        # Brand block: crisp white on solid navy, handle in mustard.
        brand = VGroup(
            label("Made by", 26, WHITE).set_opacity(0.9),
            label("Mugambi Ndwiga", 36, WHITE, bold=True),
            label("@craftsandengineering", 26, PALETTE["mustard"]),
        ).arrange(DOWN, buff=0.18)
        brand.scale_to_fit_width(min(brand.width, fw - 0.7))

        block = VGroup(accent, brand).arrange(DOWN, buff=0.5).move_to(ORIGIN)

        # Sequential reveal: draw the motif, then bring the credit up.
        self.play(Create(motif), FadeIn(tip, scale=0.4), run_time=0.6)
        self.play(FadeIn(brand, shift=UP * 0.15), run_time=0.7)
        self.wait(TIME["close"] + 0.4)

Manim Community v0.18.1

In [5]:
import glob, os
from IPython.display import Video, HTML, display

# Locate the most recently rendered mp4
candidates = sorted(
    glob.glob("media/videos/**/*.mp4", recursive=True),
    key=os.path.getmtime,
)
if not candidates:
    print("No mp4 found — did the render complete successfully?")
else:
    latest = candidates[-1]
    size_mb = os.path.getsize(latest) / 1_048_576
    print(f"Video found: {latest}  ({size_mb:.2f} MB)")

    # Inline playback
    display(Video(latest, embed=True, width=854, height=480))

    # Download link
    from IPython.display import FileLink
    display(HTML("<br><b>Download:</b>"))
    display(FileLink(latest))

Video found: media/videos/content/1920p30/SlopeStairs.mp4  (1.32 MB)


/content/media/videos/content/1920p30/SlopeStairs.mp4